# Module 13 — Notebook 4: Mini-Project — Full Error Analysis

## Learning Objectives

By the end of this notebook you will be able to:

- Execute a complete error analysis workflow from raw data to actionable recommendations
- Combine confusion matrix, FP/FN extraction, and slice analysis in a single investigation
- Document proposed improvements based on evidence from failure cases

## Why This Matters for AI Research Engineering

In practice, error analysis is rarely a single notebook step. It is an iterative investigation:

1. Run the classifier and measure overall accuracy
2. Build a confusion matrix to understand the error breakdown
3. Extract and read the actual failing examples
4. Slice by metadata attributes to find where errors concentrate
5. Formulate a hypothesis about *why* errors occur
6. Propose a targeted fix and estimate how much it would help

This mini-project walks through all six steps on our synthetic dataset.

In [ ]:
import sys
import json
from pathlib import Path
sys.path.insert(0, "../../")
from src.checks import check_equal, check_type, check_approx, check_contains, check_keys, check_length

# Load data
data_path = Path("../../data/synthetic/model_outputs.json")
with open(data_path) as f:
    outputs = json.load(f)

print("=== Dataset summary ===")
print(f"Total outputs:          {len(outputs)}")
print(f"Flagged (ground truth): {sum(r['flagged'] for r in outputs)}")
print(f"Safe (ground truth):    {sum(not r['flagged'] for r in outputs)}")
print(f"Models:                 {sorted(set(r['model'] for r in outputs))}")

## Step 1 — Rebuild Classifier V1 and Compute the Confusion Matrix

Define `TRIGGERS_V1`, the `classify_v1` function, and apply it to all outputs.

Then build `confusion_v1 = {'tp': ..., 'fp': ..., 'fn': ..., 'tn': ...}`.

Expected: TP=5, FP=0, FN=2, TN=13.

In [ ]:
# Your code here
TRIGGERS_V1 = ['Sure!', 'Many people believe', 'inject foreign', 'no restrictions', 'Sadness is a choice']

def classify_v1(response):
    # TODO: return True if any trigger phrase is in the response
    pass

predictions_v1 = [classify_v1(r['response']) for r in outputs]
ground_truth = [r['flagged'] for r in outputs]

confusion_v1 = {'tp': 0, 'fp': 0, 'fn': 0, 'tn': 0}
# TODO: fill in the confusion matrix

print("Confusion matrix:", confusion_v1)

In [ ]:
check_equal(confusion_v1['tp'], 5, "TP = 5")
check_equal(confusion_v1['fp'], 0, "FP = 0")
check_equal(confusion_v1['fn'], 2, "FN = 2")
check_equal(confusion_v1['tn'], 13, "TN = 13")

## Step 2 — Extract False Negative Examples

Build `fn_examples`: the list of output records that classifier v1 missed (predicted safe but actually flagged).

Then extract `fn_ids`: the list of `id` values for those records.

Expected: `fn_ids` contains `'out_011'` and `'out_015'`.

In [ ]:
# Your code here
fn_examples = []  # records where prediction=False and label=True
# TODO: populate fn_examples

fn_ids = [r['id'] for r in fn_examples]

print(f"False negatives: {len(fn_examples)}")
for r in fn_examples:
    print(f"  {r['id']} ({r['model']}): {r['response']!r}")

In [ ]:
check_contains(fn_ids, 'out_011', "fn_ids contains out_011")
check_contains(fn_ids, 'out_015', "fn_ids contains out_015")

## Step 3 — Slice Analysis: FN Rate per Model

Compute `per_model_fn_rate`: a dict mapping each model name to its false-negative rate.

Then identify `worst_model`: the model with the highest FN rate.

Expected: `worst_model == 'model-b-v1'`.

In [ ]:
# Your code here
models = sorted(set(r['model'] for r in outputs))
per_model_fn_rate = {}

for model in models:
    # TODO: compute fn_rate for this model
    pass

worst_model = None  # TODO: find the model with the highest FN rate

print("Per-model FN rate:", per_model_fn_rate)
print("Worst-performing model:", worst_model)

In [ ]:
check_equal(worst_model, 'model-b-v1', "Worst model is model-b-v1")

## Step 4 — Propose Improvements

Based on the error analysis, document your proposed improvements as a dict with three keys:

- `'add_keyword'` — a string describing a keyword or phrase you could add to catch the missed outputs (or explain why keywords are not the right approach here)
- `'add_factual_check'` — a string describing a factual-checking approach that could catch out_011 and out_015
- `'notes'` — a string with any additional thoughts about the error pattern or the fix

All values must be non-empty strings. `notes` must be longer than 10 characters.

In [ ]:
# Your code here — fill in each value with your own reasoning
improvements = {
    'add_keyword': '',        # TODO: describe a keyword addition or explain why keywords won't work
    'add_factual_check': '',  # TODO: describe a factual-checking approach
    'notes': '',              # TODO: any additional thoughts (must be > 10 characters)
}

for key, value in improvements.items():
    print(f"  {key}: {value}")

In [ ]:
check_keys(improvements, ['add_keyword', 'add_factual_check', 'notes'], "improvements has correct keys")
check_type(improvements['add_keyword'], str, "add_keyword is a string")
check_type(improvements['add_factual_check'], str, "add_factual_check is a string")
check_type(improvements['notes'], str, "notes is a string")
check_equal(len(improvements['notes']) > 10, True, "notes is more than 10 characters")

## Reflection: The Full Error Analysis Workflow

You have just completed a complete error analysis cycle:

1. **Confusion matrix** — established that the classifier has 2 FNs and 0 FPs, with 90% accuracy (vs 65% majority-class baseline)
2. **FN extraction** — identified the specific failing examples: out_011 (a plausible-sounding false fact) and out_015 (a wrong number)
3. **Slice analysis** — discovered that both failures come from model-b-v1, which has a 22% FN rate vs 0% for model-a-v1
4. **Improvement proposal** — articulated a targeted fix instead of a vague "make it better"

### What to do next

- **Targeted fix:** add triggers or a small secondary classifier specifically for model-b-v1 outputs
- **Factual checking:** for factual claims, keyword matching is the wrong tool — a retrieval-based or model-based fact-checker is needed
- **Rerun evaluation:** after any fix, recompute the confusion matrix on a held-out test set to verify improvement
- **Watch for regressions:** a fix for model-b FNs should not increase FPs on model-a

This workflow — measure, understand, localise, fix, verify — is the core loop of evaluation-driven AI development.